In [1]:
import os
import json
import numpy as np
import cupy as cp
import zarr
import vtk
from vtk.util import numpy_support
import pyvista as pv

alpha_low = 0.0001
alpha_high = 0.05

print("imports completed")

def load_component_data(output_dir):
    with open(os.path.join(output_dir, "render_metadata.json")) as f:
        meta = json.load(f)
    root = zarr.open_group(os.path.join(output_dir, "final_density.zarr"), mode="r")
    arrays = [np.asarray(root[ch["name"]]) for ch in meta["channels"]]
    return arrays, meta

def build_independent_components_volume(arrays, meta):
    n_components = len(arrays)
    shape = arrays[0].shape
    stacked = np.stack(arrays, axis=-1).astype(np.float32)  # (nx, ny, nz, n_components)

    image_data = vtk.vtkImageData()
    image_data.SetDimensions(shape[0], shape[1], shape[2])
    image_data.SetSpacing(1, 1, 1)
    image_data.SetOrigin(0, 0, 0)

    flat = stacked.reshape(-1, n_components, order='F')
    vtk_array = numpy_support.numpy_to_vtk(flat, deep=True, array_type=vtk.VTK_FLOAT)
    vtk_array.SetNumberOfComponents(n_components)
    image_data.GetPointData().SetScalars(vtk_array)

    mapper = vtk.vtkSmartVolumeMapper()
    mapper.SetInputData(image_data)

    volume_property = vtk.vtkVolumeProperty()
    volume_property.IndependentComponentsOn()
    volume_property.SetInterpolationTypeToLinear()

    for i, ch in enumerate(meta["channels"]):
        lo, hi = ch["contrast_limits"]
        r, g, b = ch["color"]

        color_tf = vtk.vtkColorTransferFunction()
        color_tf.AddRGBPoint(lo, r, g, b)
        color_tf.AddRGBPoint(hi, r, g, b)  # fixed color; opacity carries density info
        volume_property.SetColor(i, color_tf)

        opacity_tf = vtk.vtkPiecewiseFunction()
        opacity_tf.AddPoint(lo, alpha_low)
        opacity_tf.AddPoint(hi, alpha_high)
        volume_property.SetScalarOpacity(i, opacity_tf)

    volume = vtk.vtkVolume()
    volume.SetMapper(mapper)
    volume.SetProperty(volume_property)
    return volume

print("Fin")

/home/nehadesigar/pixi_env/.pixi/envs/default/lib/python3.12/site-packages/cupy/_environment.py:670: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


imports completed
Fin


In [2]:
output_dir = "OUTPUTS/3D_A_16_B_20"

In [3]:
def render_static_image(output_dir, save_path, azimuth=0, elevation=0, roll=0,
                          window_size=(1000, 1000)):
    """
    azimuth: rotation around the vertical axis, degrees
    elevation: rotation up/down from the horizontal, degrees
    roll: rotation of the camera about its own view axis, degrees
    """
    arrays, meta = load_component_data(output_dir)
    volume = build_independent_components_volume(arrays, meta)

    plotter = pv.Plotter(off_screen=True, window_size=window_size)
    plotter.add_actor(volume)
    plotter.set_background("white")

    plotter.camera_position = 'iso'   # default reference orientation
    plotter.camera.azimuth = azimuth
    plotter.camera.elevation = elevation
    plotter.camera.roll = roll

    plotter.screenshot(save_path)
    plotter.close()

render_static_image(output_dir,
    "OUTPUTS/3D_A_16_B_20/render.png",
    azimuth=0, elevation=0, roll=0,
)

print("Fin")

Fin


2026-07-28 14:06:08.901 (   2.250s) [    78D02CDD9740]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=


In [4]:
arrays, meta = load_component_data(output_dir)
print(arrays[0].min())
print(arrays[0].max())
print(arrays[0].shape)

def print_density_range(output_dir, component_name="component_0"):
    z = zarr.open(f"{output_dir}/final_density.zarr", mode="r")
    rho = np.asarray(z[component_name])
    print(f"min: {rho.min()}, max: {rho.max()}")

print_density_range("OUTPUTS/3D_A_16_B_20")

4.1528497e-06
0.00062601286
(128, 128, 128)
min: 4.152849669480929e-06, max: 0.0006260128575377166


In [5]:
def render_rotation_video(output_dir, save_path, n_frames=120, fps=24,
                            elevation=0, window_size=(1000, 1000)):
    arrays, meta = load_component_data(output_dir)
    volume = build_independent_components_volume(arrays, meta)

    plotter = pv.Plotter(off_screen=True, window_size=window_size)
    plotter.add_actor(volume)
    plotter.set_background("white")

    plotter.camera_position = 'iso'
    plotter.camera.elevation = elevation

    plotter.open_movie(save_path, framerate=fps)
    step = 360.0 / n_frames
    for _ in range(n_frames):
        plotter.camera.azimuth += step
        plotter.write_frame()
    plotter.close()

render_rotation_video(output_dir,
    "OUTPUTS/3D_A_16_B_20/rotation.mp4",
    n_frames=360, fps=36,
)

print("Fin")

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1000, 1000) to (1008, 1008) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Fin


In [6]:
import zarr
z = zarr.open_group("OUTPUTS/3D_A_16_B_20/final_density.zarr", mode="r")
print(list(z.array_keys()))          # confirms what datasets actually exist
arr = z["component_0"][:]             # or whatever name shows up above
print(arr.shape, arr.dtype)
print(arr.min(), arr.max())
print(arr[arr.shape[0]//2, arr.shape[1]//2, arr.shape[2]//2])  # spot-check a middle voxel

['component_0', 'component_1']
(128, 128, 128) float32
4.1528497e-06 0.00062601286
4.2372976e-06


In [7]:
import json
with open("OUTPUTS/3D_A_16_B_20/render_metadata.json") as f:
    print(json.load(f))

{'n_components': 2, 'gridpoints': 128, 'channels': [{'name': 'component_0', 'color': [1, 0, 0], 'contrast_limits': [4.152849669480929e-06, 0.0006260128575377166]}, {'name': 'component_1', 'color': [0, 1, 0], 'contrast_limits': [7.571458809252363e-06, 0.00019494828302413225]}], 'sim_params': {'B2aa': 1121, 'B2ab': 1467, 'B2bb': 1878, 'valence': 4, 'Ka': 1000000.0, 'Kb': 1000000.0, 'recent_step': 16000000, 'max_A': 0.0006260128792420937, 'min_A': 4.1528497043324225e-06, 'max_B': 0.0001949482891901415, 'min_B': 7.571458615008373e-06}}
